In [ ]:
%reload_ext autotime
import geopandas as gpd
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.cluster import KMeans
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from shapely import box
pd.set_option('display.max_columns', 500)

In [ ]:
df = gpd.read_parquet("../data/areas+NDVI.parquet").reset_index(drop=True)
df

## Weighted k-means

Each polygon has many features, but they are spread very unevenly across the variables they
describe: every terrain attribute contributes 15 columns (5 statistics × old/new/diff), NDVI 15,
each hydrological index 5, and `distance_to_river` just one. After standardisation every column
has unit variance, so plain k-means weights each variable by its number of columns, and the
120 terrain columns dominate the clustering.

To correct for this, the standardised columns are grouped by the variable they describe and each
column is multiplied by $\sqrt{w_g / n_g}$, where $n_g$ is the number of (non-constant) columns in
group $g$ and $w_g$ its weight. Each group then contributes $w_g$ to the total variance, so with the default
$w_g = 1$ every variable has the same influence on the squared Euclidean distances k-means uses.
Change `GROUP_WEIGHTS` to emphasise a variable, or set its weight to 0 to leave it out.

The LCDB 2018 land-cover class and the GNS rock group are categories, so they are one-hot encoded rather
than treated as numbers. Processing flags (`tile`, `split_by_river`) are not used as features.

In [ ]:
STATS = ["min", "max", "mean", "median", "std"]
TERRAIN = ["roughness", "slope", "aspect", "curvature", "terrain_ruggedness_index",
           "rugosity", "profile_curvature", "planform_curvature"]
HYDRO = ["downslope_dist", "elevation_above_stream", "depth_to_water", "twi", "spi", "connectivity_index"]
LOG_TRANSFORM = ["area", "rock_age"] + [f"spi_{s}" for s in STATS]  # heavily right-skewed, non-negative

def stat_cols(*prefixes):
    return [f"{p}_{s}" for p in prefixes for s in STATS]

# Land cover: one-hot encoded LCDB class plus the wetland/onshore flags
land_cover = pd.get_dummies(df["Class_2018"].astype("string"), prefix="lcdb", dtype=float)
for col in ["Wetland_18", "Onshore_18"]:
    land_cover[col] = df[col].astype("boolean").fillna(False).astype(float)
# Geology: one-hot encoded GNS rock group (e.g. mudstone, alternating sandstone/mudstone)
rock_type = (pd.get_dummies(df["rock_group"].astype("string"), prefix="rock", dtype=float)
             if "rock_group" in df else pd.DataFrame(index=df.index))
# 2023 land cover is after the change, so it's off by default (see GROUP_WEIGHTS)
land_cover_2023 = (pd.get_dummies(df["Class_2023"].astype("string"), prefix="lcdb23", dtype=float)
                   if "Class_2023" in df else pd.DataFrame(index=df.index))
categorical = pd.concat([land_cover, rock_type, land_cover_2023], axis=1)

feature_groups = {
    "elevation": stat_cols("old", "new"),
    "elevation_change": stat_cols("diff"),
    **{a: stat_cols(f"old_{a}", f"new_{a}", f"diff_{a}") for a in TERRAIN},
    **{h: stat_cols(h) for h in HYDRO},
    "ndvi": stat_cols("NDVI_old", "NDVI_new"),
    "ndvi_change": stat_cols("NDVI_diff"),
    "area": ["area"],
    "distance_to_river": ["distance_to_river"],
    "in_river": ["in_river"],
    "land_cover": list(land_cover.columns),
    "land_cover_2023": list(land_cover_2023.columns),
    "rock_type": list(rock_type.columns),
    "rock_age": ["rock_age"],
    "mapped_landslide": ["landslide_share"],
}
# Groups not listed get weight 1, e.g. {"elevation_change": 2, "aspect": 0}. 2023 land cover
# (LCDB v6.0) was mapped after the change, so it describes the outcome; set its weight above 0 to use it.
GROUP_WEIGHTS = {"land_cover_2023": 0}

features = pd.concat([df.drop(columns=categorical.columns.intersection(df.columns)), categorical], axis=1)
for group, cols in list(feature_groups.items()):
    present = [c for c in cols if c in features.columns]
    if len(present) < len(cols):
        print(f"{group}: {len(cols) - len(present)} of {len(cols)} columns missing")
    if present and GROUP_WEIGHTS.get(group, 1) > 0:
        feature_groups[group] = present
    else:
        del feature_groups[group]
columns = [c for cols in feature_groups.values() for c in cols]
X = features[columns].astype(float)
log_cols = X.columns.intersection(LOG_TRANSFORM)
X[log_cols] = np.log1p(X[log_cols])

# k-means needs complete rows
complete = X.notna().all(axis=1)
print(f"Dropping {(~complete).sum()} of {len(X)} polygons with missing values; most common gaps:")
print(X.isna().sum().sort_values(ascending=False).head(5).to_string())
df, X = df[complete].reset_index(drop=True), X[complete].reset_index(drop=True)

In [ ]:
scaled = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)
summary = []
for group, cols in feature_groups.items():
    weight = GROUP_WEIGHTS.get(group, 1)
    varying = max(int((X[cols].std(ddof=0) > 0).sum()), 1)  # constant columns carry no variance
    scaled[cols] *= np.sqrt(weight / varying)
    summary.append((group, len(cols), weight, scaled[cols].var(ddof=0).sum()))
summary = pd.DataFrame(summary, columns=["group", "columns", "weight", "variance"]).set_index("group")
summary["share_of_distance"] = summary.variance / summary.variance.sum()
summary

In [ ]:
clusters = KMeans(n_clusters=25, n_init=10, random_state=0).fit_predict(scaled)
unique_values, counts = np.unique(clusters, return_counts=True)
plt.bar(unique_values.astype(str), counts)

In [ ]:
df["cluster"] = clusters

In [ ]:
df.to_parquet("../data/clusters.parquet")

In [ ]:
df.to_file("../data/clusters.gpkg")

In [ ]:
bbox = box(2053600, 5802000, 2058400, 5809200)
subset = df[df.within(bbox)]
display(subset.cluster.value_counts())
print(subset.shape)
subset.explore("cluster", legend=True, tooltip=["cluster", "area", "distance_to_river"] + df.columns[df.columns.str.startswith("diff")].tolist(), categorical=True, tiles="ESRI.WorldImagery")